<a href="https://colab.research.google.com/github/acoiman/pdt/blob/main/asthma_mortality/notebooks/Python/05.Asthma_Mortality_AP_PM10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏭 Air Pollution Data ($PM_{10}$)

In this notebook, we will calculate the mean concentration ($μg/m^3$) of fine particulate matter with a diameter of 10 micrometers or less $PM_{10}$ by department in Argentina from 2001 to 2022. We obtained the $PM_{10}$ data form [CONAE](https://www.argentina.gob.ar/ciencia/conae) (Comisión Nacional de Actividades Espaciales)

##📦 Import Required Libraries

In [ ]:
# geospatial data handling
import rasterio
from rasterio.mask import mask
import geopandas as gpd
import geemap
import ee
import rioxarray
import numpy as np

# array data handling
import xarray as xr

# dataframe handling
import pandas as pd

# plotting
import matplotlib.pyplot as plt
import seaborn as sns

# other libraries
import os
import glob
import zipfile
import shutil
from pathlib import Path
import subprocess
from itables import init_notebook_mode

In [ ]:
# Set the PROJ_LIB path
os.environ['PROJ_LIB'] = "/opt/conda/envs/gds/share/proj"

In [ ]:
# change directory to work folder (at the begining,  docker container enter into /home/jovyan/)
%cd work

In [ ]:
 # Pandas Copy-on-Write
 pd.options.mode.copy_on_write = True

## 👩🏻‍💻Calculating Annual Mean $PM_{10}$ per year

<font color="red">This is an informative section, and running the bellow cells are not necessary to execute the rest of the notebook. You can proceed directly to the `Merging Calculated PM10 Dataset with Other Features` section. However, if you want to generate the annual mean $PM_{10}$  values for each year, read the texts and run the cells below.</font>

The monthly data required to run the code in this section are not stored on Google Drive because of their large file size. To obtain these data, follow the instructions provided in the [ARGeoPM10_PM10_CONAE](https://github.com/CONAE-GVT/ARGeoPM10_PM10_CONAE) repository.</p>

Once you obtain the daily $PM_{10}$ data, calculate the monthly mean values and store them in your working directory inside subfolders named `predict_year`, where `year` corresponds to each year from 2002 to 2022. Also, copy the folder `pdt/asthma_mortality/data/shp/` into your working directory. With this data, you can follow along with the execution below.

We will calcualte the annual mean concentration of $PM_{10}$ (particulate matter with a diameter of up to 10 micrometers), expressed in micrograms per cubic meter ($\mu g/m^3$), using modeled  $PM_{10}$  data by Garcia et al. [1]. These authors developed a modeling system called "Empatia"  to generate geospatial datasets of daily and monthly surface concentrations of $PM_{10}$ across Argentina from multisource geospatial data and Random Forest (RF) [2].

Through  the Empatia server, we get the daily and monthly data to calculate the annual mean concentration of $PM_{10}$  from 2002 to 2022. Data were only available from July 2002 onward because the modelling system could not process data for earlier months and years. The system generated daily and monthly predicted $PM_{10}$ concentrations from MODIS Terra (13:30 UTC) and MODIS Aqua (16:30 UTC).

 The following code blocks allow us to calculate the mean annual concentration of $PM_{10}$ ($\mu g/m^3$). We opted to use only data from MODIS Terra (13:30 UTC) because aerosol concentrations usually peak during the morning hours [3].

In [ ]:
# function to find path
def find_path(name):
    for root, dirs, files in os.walk('.'):
        if name in files or name in dirs:
            return os.path.join(root, name)
    return None

In [ ]:
# Dynamically locate the shapefile of Argenina boundariess
boundary_path = find_path("ar_poly.shp")

if not boundary_path:
    print(f"❌ Error: Could not find shapefile 'ar_poly.shp'")
else:
    print(f"Found shapefile at: {boundary_path}")
    argentina = gpd.read_file(boundary_path).to_crs("EPSG:4326")

In [ ]:
# list to store the number of images used to calculate the mean PM10 values per year
total_files_list = []

# Iterate through each year from 2002 to 2022
for year in range(2002, 2023):
    target_folder = f"predict_{year}"
    # Dynamically find the path for the specific year folder
    base_dir = find_path(target_folder)

    if not base_dir:
        print(f"⚠️ Folder {target_folder} not found. Skipping...")
        continue

    print(f"\n--- Processing Year: {year} ---")

    # Locate subfolders containing '133000' and find .tif/.TIFF files
    search_pattern = os.path.join(base_dir, "*133000*", "*.tif")
    tiff_files = glob.glob(search_pattern) + glob.glob(os.path.join(base_dir, "*133000*", "*.TIFF"))

    clipped_rasters = []

    # Process each file for the current year
    for tiff_path in tiff_files:
        try:
            # Open with masking (NoData -> NaN)
            rds = rioxarray.open_rasterio(tiff_path, masked=True)

            # Select Band 1: PM10_media_Terra
            band1 = rds.sel(band=1)

            # Clip using the shapefile
            clipped = band1.rio.clip(argentina.geometry, argentina.crs, drop=True)
            clipped_rasters.append(clipped)
        except Exception as e:
            print(f"Error processing {tiff_path}: {e}")

    # Calculate the average for the year
    if clipped_rasters:
        # Stack along a new dimension to compute mean across space/time
        combined = xr.concat(clipped_rasters, dim="time")
        mean_raster = combined.mean(dim="time", skipna=True)

        # Save with the requested naming convention
        output_filename = f"PM10_AR_CONAE_{year}.tif"
        mean_raster.rio.to_raster(output_filename)
        print(f"✅ Saved: {output_filename} (from {len(clipped_rasters)} files)")
        total_files_list.append(len(clipped_rasters))
    else:
        print(f"❌ No files processed successfully for {year}.")

total_files = sum(total_files_list)
print(f"\nTotal files processed: {total_files}")

We observe that, for almost all years except 2002 and 2014, 10 to 12 images were used to calculate the annual mean concentration of $PM_{10}$. In total, 233 images were processed.

## 🌍 Connect to Google Earth Engine (GEE)

In [ ]:
# trigger the authentication flow
ee.Authenticate()

In [ ]:
# initialize the library.
# ee.Initialize(project='ee-pdt')
ee.Initialize(project='gee-project-419000')# change to your project ID
print(ee.String('Hello from the Earth Engine servers!').getInfo())

## 🧮 Calculate mean $PM_{10}$ per department/year (2002–2022)

### Upload  data to GEE

The calculated annual mean $PM_{10}$ per year were uploaded to Google Earth Engine through the Code Editor interface.

### Calculating mean $PM_{10}$ per department/year

Through the following function, we will batch-extract the annual mean $PM_{10}$ data for each department. The output will then be merged with the dataset containing the asthma mortality rate.

In [ ]:
def calculate_and_merge_pm10_by_year(start_year=2002, end_year=2022,
                                     asset_image_prefix="projects/ee-pdt/assets/CONAE_MOD_CDA_ARGeoPM10_PM10m_2002_2022_133000/CONAE_MOD_CDA_ARGeoPM10_PM10m_",
                                     fc_asset="projects/ee-pdt/assets/argentina/ar_dpto",
                                     output_folder="pdt/asthma_mortality/data/gpkg/",
                                     merged_filename="pm10_2002_2022.gpkg"):

    # Load FeatureCollection containing Argentina departments
    fc = ee.FeatureCollection(fc_asset)

    merged_gdf = None  # To accumulate results

    for year in range(start_year, end_year + 1):
        print(f"Processing year {year}...")

        # Load PM10 image
        image_path = f"{asset_image_prefix}{year}"
        pm10_image = ee.Image(image_path)

        # Reduce regions to get the mean
        reduced_fc = pm10_image.reduceRegions(
            collection=fc,
            reducer=ee.Reducer.mean(),
            scale=1000
        )

        # Convert to GeoDataFrame
        gdf = geemap.ee_to_gdf(reduced_fc)

        # Rename and round
        gdf = gdf.rename(columns={'mean': f'PM10_{year}'})
        gdf[f'PM10_{year}'] = gdf[f'PM10_{year}'].round(2)

        # Keep IDDPTO, geometry, and PM10 related columns
        if merged_gdf is None:
            merged_gdf = gdf.copy()
        else:
            merged_gdf[f'PM10_{year}'] = gdf[f'PM10_{year}']

    # Save merged gpkg
    output_path = os.path.join(output_folder, merged_filename)
    merged_gdf.to_file(output_path, encoding='utf-8')
    print(f"\nFile saved at: {output_path}")

In [ ]:
# aply the function to get annual mean PM10 data for each department
calculate_and_merge_pm10_by_year()

### Merging calculated $PM_{10}$ dataset with other features

In this section, we will merge the calculated $PM_{10}$  data with the dataset containing $PM_{2.5}$  measurements. Because $PM_{10}$  data are not available for 2001, values for that year will be imputed using the corresponding values from 2002.

In [ ]:
# Load local pm10 data
gdf_pm10 = gpd.read_file("pdt/asthma_mortality/data/gpkg/pm10_2002_2022.gpkg")

In [ ]:
# drop geometry column
gdf_pm10 = gdf_pm10.drop(columns=['geometry'])

In [ ]:
# Display the first few rows of the DataFrame
init_notebook_mode(all_interactive=True)
gdf_pm10

In [ ]:
# Load `gdf` from a gpkg file containing AMR+PM2.5 from 2001 to 2022.
gdf = gpd.read_file("pdt/asthma_mortality/data/gpkg/tma_pm25_2001_2022.gpkg")

In [ ]:
# Display the first few rows of the DataFrame
init_notebook_mode(all_interactive=True)
gdf.head()

In [ ]:
# Merge AMR+PM25 gdf and PM10 gdf based on 'IDDPTO', preserving all data from AMR gdf
gdf_pm10_tma  = gdf.merge(gdf_pm10, on='IDDPTO', how='left')

In [ ]:
# Display the first few rows of the merged GeoDataFrame
init_notebook_mode(all_interactive=True)
gdf_pm10_tma.head()

In [ ]:
# impute values of PM10 in 2002 to 2001 (missing year)
gdf_pm10_tma['PM10_2001'] = gdf_pm10_tma['PM10_2002']

In [ ]:
# Display the first few rows of the merged GeoDataFrame
init_notebook_mode(all_interactive=True)
gdf_pm10_tma.head()

In [ ]:
# Remove the persistent credential file of GEE
cred_path = Path.home() / '.config' / 'earthengine'
if cred_path.exists():
    shutil.rmtree(cred_path)
    print("GEE Persistent credentials removed.")

## 📙References

[1] F. Garcia, S. Represa, L. S. Della Ceca, K. Clemoveki,  miguel calderon, and C. Scavuzzo, Nuevos productos para la calidad del aire en Argentina, derivados de información satelital: PM10 + ICA. 2022.

[2] F. Gracía Ferreira, “ARGeoPM10_PM10_CONAE/documents at master · CONAE-GVT/ARGeoPM10_PM10_CONAE,” GitHub. Accessed: May 28, 2026. [Online]. Available: https://github.com/CONAE-GVT/ARGeoPM10_PM10_CONAE/tree/master/documents

[3] J.-P. Chen, T.-S. Tsai, and S.-C. Liu, “Aerosol nucleation spikes in the planetary boundary layer,” Atmospheric Chemistry and Physics, vol. 11, no. 14, pp. 7171–7184, Jul. 2011, doi: 10.5194/acp-11-7171-2011.


